In [0]:
df_map = spark.read \
    .option("mode", "PERMISSIVE") \
    .json("/Volumes/local_cataog/default/local_volume/map_cities.json")

df_map.printSchema()

In [0]:
df_map = spark.read \
    .option("multiline", "true") \
    .option("inferSchema", "true") \
    .json("/Volumes/local_cataog/default/local_volume/map_cities.json")

display(df_map)

**Standardize the column names**

In [0]:
df_map = df_map.toDF(*[c.strip().lower() for c in df_map.columns])

**Trim whitespace in all string columns**

In [0]:
from pyspark.sql import functions as F
for c, t in df_map.dtypes:
    if t == "string":
        df_bulk = df_map.withColumn(c, F.trim(F.col(c)))

**Drop rows where the key is null**

In [0]:
df_map = df_map.filter(F.col("city_id").isNotNull())

**Remove duplicate rows**

In [0]:
df_map = df_map.dropDuplicates(["city_id"])

In [0]:
df_map.display()

**One Big Table**

In [0]:
from pyspark.sql import functions as F

base = "/Volumes/local_cataog/default/local_volume"

# 0. Check the file names first
display(dbutils.fs.ls(base))

def load(name, key):
    df = spark.read.option("multiLine", True).json(f"{base}/{name}.json")
    df = df.toDF(*[c.strip().lower() for c in df.columns])      # clean column names
    for c, t in df.dtypes:                                       # trim strings, empty -> null
        if t == "string":
            df = df.withColumn(c, F.nullif(F.trim(F.col(c)), F.lit("")))
    return df.filter(F.col(key).isNotNull()).dropDuplicates([key])

# 1. Load and clean the 7 files
rides    = load("bulk_rides", "ride_id")
makes    = load("map_vehicle_makes", "vehicle_make_id")
types    = load("map_vehicle_types", "vehicle_type_id")
statuses = load("map_ride_statuses", "ride_status_id")
payments = load("map_payment_methods", "payment_method_id")
reasons  = load("map_cancellation_reasons", "cancellation_reason_id")
cities   = load("map_cities", "city_id").select(
              "city_id", F.col("city").alias("pickup_city"), "state", "region")

# 2. Fix data types on rides
rides = (rides
    .withColumn("booking_timestamp", F.to_timestamp("booking_timestamp"))
    .withColumn("pickup_timestamp",  F.to_timestamp("pickup_timestamp"))
    .withColumn("dropoff_timestamp", F.to_timestamp("dropoff_timestamp")))

# 3. Join everything into one big table
obt = (rides
    .join(makes,    "vehicle_make_id",        "left")
    .join(types,    "vehicle_type_id",        "left")
    .join(statuses, "ride_status_id",         "left")
    .join(payments, "payment_method_id",      "left")
    .join(reasons,  "cancellation_reason_id", "left")
    .join(cities,   rides.pickup_city_id == cities.city_id, "left")
    .drop("city_id"))

# 4. Row count check
assert obt.count() == rides.count(), "Row count changed: a lookup has duplicate keys"

# 5. Save
(obt.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("local_cataog.default.rides_obt"))

display(spark.table("local_cataog.default.rides_obt"))